# Credit Card Approval Prediction - Epic 3: Data Preprocessing

This notebook represents the Preprocessing Pipeline. We clean the dataset, merge applicant records with credit history logs, engineer demographic and credit summary features, encode categorical columns, and scale features for ML algorithms.

## 1. Import Core Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings

warnings.filterwarnings('ignore')
print("Preprocessing libraries loaded.")

## 2. Load Datasets and Drop Duplicates

We drop any exact duplicate records in both application and credit dataframes.

In [ ]:
app_df = pd.read_csv(os.path.join('..', '03_Dataset', 'application_record.csv'))
credit_df = pd.read_csv(os.path.join('..', '03_Dataset', 'credit_record.csv'))

print(f"Application count before duplicate removal: {len(app_df)}")
app_df.drop_duplicates(inplace=True)
print(f"Application count after duplicate removal: {len(app_df)}")

print(f"Credit logs before duplicate removal: {len(credit_df)}")
credit_df.drop_duplicates(inplace=True)
print(f"Credit logs after duplicate removal: {len(credit_df)}")

## 3. Handle Missing Values

We fill any missing occupation values with 'Unknown' to maintain dataset length.

In [ ]:
print("Missing values in Application:")
print(app_df.isnull().sum())

app_df['OCCUPATION_TYPE'].fillna('Unknown', inplace=True)
print(f"Missing values after treatment: {app_df['OCCUPATION_TYPE'].isnull().sum()}")

## 4. Basic Demographic Cleaning

Convert birth days and employment days to positive years and drop the raw negative columns.

In [ ]:
app_df['Age_Years'] = -app_df['DAYS_BIRTH'] / 365.25
app_df['Employed_Years'] = app_df['DAYS_EMPLOYED'].apply(lambda x: 0.0 if x > 0 else -x / 365.25)

app_df.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL'], inplace=True)
print(app_df[['Age_Years', 'Employed_Years']].describe())

## 5. Aggregate Credit Records and Construct Target Label

We group the credit logs by `ID` and classify applicants as Approved (`0`) or Rejected (`1`) based on their repayment history. A status of `1`, `2`, `3`, `4`, or `5` indicates delinquency and will result in rejection.

In [ ]:
# Map status to numeric risk value
status_risk_map = {'X': 0, 'C': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
credit_df['Risk_Score'] = credit_df['STATUS'].map(status_risk_map)

# Aggregate on ID
credit_summary = credit_df.groupby('ID').agg(
    Credit_Window=('MONTHS_BALANCE', lambda x: x.max() - x.min()),
    Max_Delay=('Risk_Score', 'max'),
    Num_Records=('MONTHS_BALANCE', 'count')
).reset_index()

# Define target: 1 if Max_Delay >= 1 else 0
credit_summary['y'] = (credit_summary['Max_Delay'] >= 1).astype(int)

print("Target Distribution:")
print(credit_summary['y'].value_counts())
print(credit_summary['y'].value_counts(normalize=True))

## 6. Merge Datasets

In [ ]:
merged_df = pd.merge(app_df, credit_summary[['ID', 'y']], on='ID', how='inner')
print(f"Merged Dataset Shape: {merged_df.shape}")
merged_df.drop(columns=['ID'], inplace=True) # Drop ID column as it is not an ML feature

## 7. Categorical Encoding

We use LabelEncoder to map categorical text categories to numerical levels.

In [ ]:
categorical_cols = [
    'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 
    'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE'
]

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    merged_df[col] = le.fit_transform(merged_df[col].astype(str))
    encoders[col] = le
    print(f"Encoded {col}: {list(le.classes_)} -> {list(range(len(le.classes_)))}")

# Save encoders dictionary
joblib.dump(encoders, 'encoder.pkl')
print("Encoders saved successfully.")

## 8. Feature Scaling

Scale features to normalize numeric ranges before training.

In [ ]:
X = merged_df.drop(columns=['y'])
y = merged_df['y']

feature_names = X.columns.tolist()
print("Features list:", feature_names)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to dataframe to save
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_names)
X_scaled_df['y'] = y.values

# Save scaler and processed dataframe
joblib.dump(scaler, 'scaler.pkl')
X_scaled_df.to_csv('processed_dataset.csv', index=False)
print("Scaler and processed dataset saved successfully.")

## 9. Train-Test Split Verification

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"Training target distribution:\n{y_train.value_counts(normalize=True)}")

## 10. Visualization of Preprocessed Data

In [ ]:
plt.figure(figsize=(6, 5))
sns.countplot(x=y, palette='Set1')
plt.title('Target Variable Distribution (y)', fontsize=14, fontweight='bold', color='#1E3A8A')
plt.xlabel('Approval Status (0=Approved, 1=Rejected)', fontsize=12)
plt.ylabel('Count', fontsize=12)
out_img = os.path.join('..', '11_Outputs', 'target_distribution.png')
plt.savefig(out_img, dpi=150, bbox_inches='tight')
plt.show()